# 08 · Özgün model: ablasyonlar (3 seed) ve McNemar

**Önce GPU açın** (T4). 5 varyant × 3 seed (42, 123, 2026), hepsi özgün modelin Optuna hiperparametreleriyle:

| Varyant | Ne çıkarıldı |
|---|---|
| full | hiçbir şey |
| - çok ölçekli birleştirme | yalnızca son aşama özelliği |
| - satır dikkati | RowAttention yok |
| - ordinal kayıp | yerine CE+sampler |
| - girişte kontrast standardizasyonu | per-image standardizasyon yok |

- 15 koşu (`Orig_full__opt` seed 42 zaten var), toplam 2-2,5 saat. Biten koşular atlanır, oturum kopsa aynı notebook'u yeniden çalıştırın
- Yalnızca **validation**. McNemar: tam model ile en iyi ablasyon, seed başına

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 optuna==5.0.0
import torch
assert torch.cuda.is_available(), 'GPU yok! Çalışma zamanı > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## Koşular

In [ ]:
from src import ablation as A
A.run_ablations(seeds=(42, 123, 2026))

## Özet

In [ ]:
import json
df = A.summarize(seeds=(42, 123, 2026))
show = df[['variant','n_seeds','param_count','val_macro_f1_mean','val_macro_f1_std','delta_f1_vs_full',
           'val_qwk_mean','val_mae_grade_mean','val_distant_error_rate_mean','val_accuracy_mean']]
display(show.round(4))
print(json.dumps(json.load(open(C.OUTPUT_DIR / 'ablation_mcnemar.json')), indent=1, ensure_ascii=False))

`outputs/results.csv`, `outputs/ablation_summary.csv` ve `outputs/ablation_mcnemar.json` dosyalarını paylaşın.